# Hello World: Remote Data Ingestion

**The 1-Minute LakeLogic Demo.**

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakelogic/LakeLogic/blob/main/examples/01_quickstart/01_hello_world.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/01_quickstart/01_hello_world.ipynb)

## ⚡ Zero Setup, Pure Value
In this example, we'll demonstrate how LakeLogic can point at a **remote URL** and instantly deliver a governed Data Lakehouse table. 

No local databases, no file downloads, just pure extraction and validation.

## Setup

In [ ]:
# SETUP: Install lakelogic if not already present
from pathlib import Path
import os
import importlib.util
import sys
import sqlite3

# --- Package check (works on Colab, Databricks, local) ---
if importlib.util.find_spec("lakelogic") is None:
    print("📦 Installing lakelogic...")
    !pip install lakelogic
    print("✅ lakelogic installed.")
else:
    print("✅ lakelogic is already installed.")

# --- Colab: clone repo so contract files (YAML, etc.) are available ---
if 'google.colab' in sys.modules:
    repo_dir = Path('/content/LakeLogic')
    if not repo_dir.exists():
        print("📂 Cloning LakeLogic repo for example files...")
        !git clone --quiet https://github.com/lakelogic/LakeLogic.git /content/LakeLogic
    os.chdir(repo_dir / 'examples' / '01_quickstart')
    print(f"📍 Working directory set to: {Path.cwd()}")


def find_example_dir(name: str) -> Path:
    cwd = Path.cwd()
    for base in [cwd] + list(cwd.parents):
        for candidate in [
            base / name,
            base / "examples",
            base / "lakelogic" / "examples"
        ]:
            if candidate.exists():
                return candidate
    return cwd


# Global Path Resolver
def get_full_path(filename: str) -> str:
    path_obj = find_example_dir(filename)
    if path_obj.is_dir():
        return str((path_obj / filename).resolve())
    return str(path_obj.resolve())

In [ ]:
# %pip install lakelogic[all]
from lakelogic import DataProcessor
import os

# Use a public raw data URL
REMOTE_URL = "https://raw.githubusercontent.com/lakelogic/LakeLogic/main/examples/01_quickstart/files/excel/data/employees.csv"

print(f"🌍 Target: {REMOTE_URL}")

## 🧪 1. Run via In-Memory Contract (Python Dict)
LakeLogic allows you to define contracts as **Python Dictionaries**. 

**Best For**: Prototyping, Dynamic rule generation, and "all-in-one" notebooks.

In [ ]:
contract_dict = {
    "version": "1.0.0",
    "dataset": "remote_employees",
    "source": {"type": "landing"},
    "quality": {
        "row_rules": [
            {"name": "Valid Email", "sql": "email LIKE '%@%'"}
        ]
    }
}

# Initialize with the dictionary directly
processor = DataProcessor(contract=contract_dict)
result = processor.run_source(REMOTE_URL)

print(f"✅ Success! Raw records: {len(result.raw)}, Valid: {len(result.good)}")

## 📝 2. Run via External Contract (YAML File)
For production, we recommend storing contracts as **YAML files**. 

**Best For**: Version Control (Git), Shared team definitions, and Formal Governance.

In [ ]:
import yaml

# 1. Save our contract to a file (simulating a git-tracked file)
with open("users_contract_remote.yaml", "w") as f:
    yaml.dump(contract_dict, f)

# 2. Initialize LakeLogic by passing the PATH string
processor_prod = DataProcessor(contract="users_contract_remote.yaml")
result_prod = processor_prod.run_source(REMOTE_URL)

print("✅ Success via YAML Path!")

## 📊 3. Inspect Results
Regardless of how you load the contract, LakeLogic returns the same rich `ValidationResult` object.

In [ ]:
print(" RAW DATA:")
display(result.raw)

In [ ]:
print("🏆 CLEAN DATA (Validated & Ready for Analytics):")
display(result.good)

In [ ]:
print("❌ BAD DATA (Invalidated & Ready for Quarantine/Review/Fix/Reprocessing):")
display(result.bad)

## 🏁 Summary: Dictionary vs. YAML

| Approach | Best Use Case | Benefit |
| :--- | :--- | :--- |
| **Python Dict** | Prototyping / Notebooks | Fast iteration, no extra files |
| **YAML File** | Production / Enterprise | Git versioning, shared governance |